# R2-Dreamer Habitat Baseline — Evaluation

**Goal:** Visualize evaluation results from R2-Dreamer on HM3D ObjectNav with top-down semantic trajectory maps.

| Property | Value |
|----------|-------|
| Agent | R2-Dreamer (JAX, no goal conditioning) |
| Environment | HM3D ObjectNav (train split, 64×64 RGB) |
| Input | `output/runs/r2dreamer-habitat-baseline/*/eval_results.json` |
| Maps | Top-down navmesh + semantic overlay + agent trajectory |

Results are produced by `python -m src.main --mode eval`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 4)})

In [ ]:
# --- Load evaluation results ---
EVAL_PATH = Path("../../../output/runs/r2dreamer-habitat-baseline/sanity-50k/eval_results.json")
assert EVAL_PATH.exists(), f"Not found: {EVAL_PATH}\nRun: uv run python -m src.main --mode eval --checkpoint <ckpt> --output_dir <path>"

with open(EVAL_PATH) as f:
    data = json.load(f)

results = data["results"]
print(f"Loaded {len(results)} episodes from {EVAL_PATH}")
print(f"Agent: {data.get('meta', {}).get('agent', 'unknown')}")

# Check if trajectory data is available
has_trajectories = "trajectory" in results[0] if results else False
print(f"Trajectory data: {'yes' if has_trajectories else 'no (re-run the eval)'}")

## Summary Statistics

In [ ]:
successes = [r["success"] for r in results]
rewards = [r["reward"] for r in results]
steps = [r["steps"] for r in results]
spls = [r["spl"] for r in results]

print(f"Episodes:     {len(results)}")
print(f"Success rate: {np.mean(successes)*100:.1f}%")
print(f"Mean SPL:     {np.mean(spls):.3f}")
print(f"Mean reward:  {np.mean(rewards):.2f} (std={np.std(rewards):.2f})")
print(f"Mean steps:   {np.mean(steps):.0f} (median={np.median(steps):.0f})")

# Per-category breakdown if available
if "object_category" in results[0]:
    from collections import defaultdict
    cat_stats = defaultdict(list)
    for r in results:
        cat_stats[r["object_category"]].append(r)
    print(f"\nPer-category ({len(cat_stats)} categories):")
    for cat, eps in sorted(cat_stats.items(), key=lambda x: -len(x[1])):
        sr = np.mean([e["success"] for e in eps]) * 100
        mr = np.mean([e["reward"] for e in eps])
        print(f"  {cat:20s}: {len(eps):3d} eps, SR={sr:5.1f}%, reward={mr:.2f}")

## Per-Episode Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

eps = range(len(results))

axes[0].bar(eps, rewards, color=["green" if s else "gray" for s in successes])
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total Reward")
axes[0].set_title("Episode Rewards")
axes[0].grid(True, alpha=0.3, axis="y")

axes[1].bar(eps, steps, color=["green" if s else "gray" for s in successes])
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Steps")
axes[1].set_title("Episode Lengths")
axes[1].axhline(y=500, color="red", linestyle="--", alpha=0.5, label="max steps")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

axes[2].bar(eps, successes)
axes[2].set_xlabel("Episode")
axes[2].set_ylabel("Success")
axes[2].set_title(f"Success (rate={np.mean(successes)*100:.0f}%)")
axes[2].set_ylim(-0.1, 1.1)

fig.suptitle("Evaluation: Per-Episode Results", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Action Distribution

In [ ]:
ACTIONS = {"STOP": 0, "MOVE_FORWARD": 1, "TURN_LEFT": 2, "TURN_RIGHT": 3}

total_counts = {name: 0 for name in ACTIONS}
for r in results:
    for name, count in r["action_counts"].items():
        total_counts[name] += count

total = sum(total_counts.values())

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(total_counts.keys(), total_counts.values())

for bar, count in zip(bars, total_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.01,
            f"{count/total*100:.1f}%", ha="center", fontsize=9)

ax.set_ylabel("Count")
ax.set_title("Action Distribution (greedy evaluation)")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## Top-Down Semantic Trajectory Maps

For each episode, render: navigable area (gray), semantic regions (colored), agent trajectory (blue→red gradient), start (green), stop (red), goal (gold).

In [ ]:
import habitat_sim

# Category colors — consistent palette for semantic regions
CATEGORY_COLORS = {}
_CMAP = plt.cm.tab20

def get_category_color(category_name):
    """Assign a consistent color to each semantic category."""
    if category_name not in CATEGORY_COLORS:
        idx = len(CATEGORY_COLORS) % 20
        CATEGORY_COLORS[category_name] = _CMAP(idx)[:3]
    return CATEGORY_COLORS[category_name]


def create_topdown_semantic_map(scene_path, resolution=0.05):
    """Create a top-down map with navmesh + semantic regions for a scene.

    Returns dict with 'image' (H, W, 3 float), bounds, and resolution.
    """
    # Create minimal sim just for map rendering
    backend_cfg = habitat_sim.SimulatorConfiguration()
    backend_cfg.scene_id = scene_path
    backend_cfg.load_semantic_mesh = True

    agent_cfg = habitat_sim.agent.AgentConfiguration()
    cfg = habitat_sim.Configuration(backend_cfg, [agent_cfg])
    sim = habitat_sim.Simulator(cfg)

    agent_y = sim.get_agent(0).get_state().position[1]
    pathfinder = sim.pathfinder
    bounds = pathfinder.get_bounds()
    x_min, z_min = bounds[0][0], bounds[0][2]
    x_max, z_max = bounds[1][0], bounds[1][2]

    xs = np.arange(x_min, x_max, resolution)
    zs = np.arange(z_min, z_max, resolution)
    H, W = len(zs), len(xs)

    # Layer 1: navigable area (light gray)
    image = np.ones((H, W, 3)) * 0.95  # white background
    for zi, z in enumerate(zs):
        for xi, x in enumerate(xs):
            if pathfinder.is_navigable(np.array([x, agent_y, z]), max_y_delta=0.5):
                image[zi, xi] = [0.85, 0.85, 0.85]  # navigable = light gray

    # Layer 2: semantic regions
    semantic_scene = sim.semantic_scene
    for region in semantic_scene.regions:
        cat_name = region.category.name() if region.category else "unknown"
        color = get_category_color(cat_name)
        aabb = region.aabb
        # Project AABB onto 2D grid
        rx_min, rx_max = aabb.center[0] - aabb.sizes[0]/2, aabb.center[0] + aabb.sizes[0]/2
        rz_min, rz_max = aabb.center[2] - aabb.sizes[2]/2, aabb.center[2] + aabb.sizes[2]/2
        xi_min = max(0, int((rx_min - x_min) / resolution))
        xi_max = min(W, int((rx_max - x_min) / resolution))
        zi_min = max(0, int((rz_min - z_min) / resolution))
        zi_max = min(H, int((rz_max - z_min) / resolution))
        # Only color navigable cells within the region
        for zi in range(zi_min, zi_max):
            for xi in range(xi_min, xi_max):
                if np.allclose(image[zi, xi], [0.85, 0.85, 0.85]):
                    image[zi, xi] = color

    # Layer 3: semantic objects (smaller, brighter)
    for obj in semantic_scene.objects:
        cat_name = obj.category.name() if obj.category else "unknown"
        color = get_category_color(cat_name)
        aabb = obj.aabb
        ox_min = aabb.center[0] - aabb.sizes[0]/2
        ox_max = aabb.center[0] + aabb.sizes[0]/2
        oz_min = aabb.center[2] - aabb.sizes[2]/2
        oz_max = aabb.center[2] + aabb.sizes[2]/2
        xi_min = max(0, int((ox_min - x_min) / resolution))
        xi_max = min(W, int((ox_max - x_min) / resolution))
        zi_min = max(0, int((oz_min - z_min) / resolution))
        zi_max = min(H, int((oz_max - z_min) / resolution))
        for zi in range(zi_min, zi_max):
            for xi in range(xi_min, xi_max):
                if not np.allclose(image[zi, xi], [0.95, 0.95, 0.95]):  # skip non-navigable
                    image[zi, xi] = np.clip(np.array(color) * 1.2, 0, 1)

    sim.close()

    return {
        "image": image,
        "x_min": x_min, "x_max": x_max,
        "z_min": z_min, "z_max": z_max,
        "resolution": resolution,
    }


def world_to_pixel(pos_3d, map_data):
    """Convert 3D world position [x, y, z] to pixel coords [col, row]."""
    x, z = pos_3d[0], pos_3d[2]
    col = (x - map_data["x_min"]) / map_data["resolution"]
    row = (z - map_data["z_min"]) / map_data["resolution"]
    return col, row


print("Rendering functions loaded.")

In [ ]:
assert has_trajectories, "No trajectory data — re-run eval_habitat.py (updated version with trajectory saving)"

# Cache maps per scene to avoid re-creating sim for episodes in the same scene
scene_maps = {}
N_EPISODES = min(len(results), 12)  # show up to 12 episodes

# Determine grid layout
ncols = min(4, N_EPISODES)
nrows = (N_EPISODES + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
if N_EPISODES == 1:
    axes = np.array([axes])
axes = axes.flat

for idx in range(N_EPISODES):
    r = results[idx]
    ax = axes[idx]

    scene_id = r["scene_id"]
    if scene_id not in scene_maps:
        # scene_id is the full path from Habitat
        scene_maps[scene_id] = create_topdown_semantic_map(scene_id, resolution=0.05)
    map_data = scene_maps[scene_id]

    # Draw base map
    ax.imshow(map_data["image"], origin="lower",
              extent=[map_data["x_min"], map_data["x_max"],
                      map_data["z_min"], map_data["z_max"]])

    # Draw trajectory as gradient line (blue=start → red=end)
    traj = np.array(r["trajectory"])
    xs, zs = traj[:, 0], traj[:, 2]
    points = np.column_stack([xs, zs]).reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    colors = plt.cm.coolwarm(np.linspace(0, 1, len(segments)))
    lc = LineCollection(segments, colors=colors, linewidths=1.5, alpha=0.8)
    ax.add_collection(lc)

    # Start marker (green triangle)
    start = r["start_position"]
    ax.plot(start[0], start[2], marker="^", color="lime", markersize=10,
            markeredgecolor="black", markeredgewidth=0.5, zorder=5)

    # Stop marker (red square)
    stop = r["trajectory"][-1]
    ax.plot(stop[0], stop[2], marker="s", color="red", markersize=8,
            markeredgecolor="black", markeredgewidth=0.5, zorder=5)

    # Goal markers (gold star)
    for gp in r["goal_positions"]:
        ax.plot(gp[0], gp[2], marker="*", color="gold", markersize=14,
                markeredgecolor="black", markeredgewidth=0.5, zorder=5)

    status = "SUCCESS" if r["success"] > 0 else "FAIL"
    cat = r.get("object_category", "?")
    ax.set_title(f"Ep {r['episode']} [{status}] {cat}\n"
                 f"steps={r['steps']} reward={r['reward']:.1f}", fontsize=9)
    ax.set_aspect("equal")

# Hide unused axes
for idx in range(N_EPISODES, len(list(axes))):
    axes[idx].set_visible(False)

# Legend
legend_elements = [
    mpatches.Patch(facecolor="lime", edgecolor="black", label="Start"),
    mpatches.Patch(facecolor="red", edgecolor="black", label="Stop"),
    mpatches.Patch(facecolor="gold", edgecolor="black", label="Goal"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=10)
fig.suptitle("Top-Down Semantic Trajectory Maps", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()